# NBA 3.1 — HCP Universe Identification v2

Builds the HCP Universe using an **indication-based** approach, replacing the product-based (SGLT2/GLP1) methodology.

| # | Criterion | Source | Change from v1 |
|---|-----------|--------|----------------|
| 1 | Indication-Based HCPs (Decile 2+) | `lrx_nbrx` | **New** — replaces SGLT2/GLP1 product filter |
| 2 | Kerendia Writers | `lrx_nbrx` | Simplified — same logic, new source |
| 3 | Specialty HCPs | `lrx_nbrx` | **Expanded** — NEPH + CARD + ENDO + PCP + HOSPITALIST, no MDM join |
| 4 | Targeting & BPT HCPs | `CP_KER_HCP` | Unchanged |

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_config"

In [0]:
from pyspark.sql import functions as F, Window
from pyspark.sql import DataFrame
from functools import reduce
from datetime import datetime as dt
import os

# ── Catalog / Schema ─────────────────────────────────────────────────────────
CATALOG = "ph-com-ai-us-prod-catalog-2553213839599380"
SCHEMA  = "pace_ker_in_house_schema"

# ── Lookback window (2 years) ─────────────────────────────────────────────────
LOOKBACK_START = "2024-09-01"   # Adjust as needed
LOOKBACK_END   = "2026-08-31"   # Adjust as needed

# ── Indication column to use ──────────────────────────────────────────────────
INDICATION_COL = "INDICATION_GROUP_APP1"   # 16 values, includes T1D

# ── Target specialties for Criterion 3 ───────────────────────────────────────
TARGET_SPECIALTIES = ["NEPH", "CARD", "ENDO", "PCP", "HOSPITALIST"]

# ── Decile cutoff: keep Decile 2+ (drop bottom-volume Decile 1) ───────────────
DECILE_CUTOFF = 2

# ── Output path ───────────────────────────────────────────────────────────────
OUTPUT_PATH = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_universe_v2"
)

print(f"Lookback : {LOOKBACK_START} → {LOOKBACK_END}")
print(f"Indication column : {INDICATION_COL}")
print(f"Target specialties: {TARGET_SPECIALTIES}")
print(f"Decile cutoff     : {DECILE_CUTOFF}+")
print(f"Output path       : {OUTPUT_PATH}")

In [0]:
# ── 1. lrx_nbrx: HCP-level weekly NBRx (primary source for criteria 1-3) ─────
lrx_nbrx = spark.table(f"`{CATALOG}`.`{SCHEMA}`.lrx_nbrx")

# ── 2. Filtered to lookback window ────────────────────────────────────────────
lrx_window = lrx_nbrx.filter(
    (F.col("WEEK_ENDING_DATE") >= LOOKBACK_START) &
    (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
)

# ── 3. NPI / BH_ID / PROVIDER_ID mapping (for Targeting & BPT) ───────────────
npi_provider_bhid_mapping = spark.table(
    f"`{CATALOG}`.kerendia_nba_3.SOURCE_PROVIDER_BHID_NPI_MAPPING"
)

print(f"lrx_nbrx total rows : {lrx_nbrx.count():,}")
print(f"lrx_window rows     : {lrx_window.count():,}")
print(f"NPI mapping rows    : {npi_provider_bhid_mapping.count():,}")

---
### Criterion 1 — Indication-Based HCPs (Decile 2+)

All HCPs prescribing to **T1D / T2D / CKD / HF** patients within the lookback window, excluding the lowest-volume decile.

**Logic:**
1. Filter `lrx_nbrx` to `INDICATION_GROUP_APP1 != 'OTHERS'`
2. Sum total NBRx per `PROVIDER_ID`
3. Rank into 10 deciles by total NBRx (`ntile(10)`, ascending — Decile 1 = lowest)
4. Drop Decile 1
5. Derive per-HCP indication flags (T1D / T2D / CKD / HF)

In [0]:
# ── Step 1: Filter to indication-relevant rows ─────────────────────────────────
lrx_ind = lrx_window.filter(F.col(INDICATION_COL) != "OTHERS")

# ── Step 2: Total NBRx per PROVIDER_ID ─────────────────────────────────────
hcp_nbrx = lrx_ind.groupBy("PROVIDER_ID").agg(
    F.sum("NBRX").alias("TOTAL_NBRX")
)

# ── Step 3: Assign deciles (ntile 10, ascending — 1 = lowest volume) ───────────
window_decile = Window.orderBy(F.asc("TOTAL_NBRX"))
hcp_nbrx = hcp_nbrx.withColumn("INDICATION_DECILE", F.ntile(10).over(window_decile))

# Decile distribution
print("Decile distribution (pre-filter):")
hcp_nbrx.groupBy("INDICATION_DECILE").agg(
    F.count("*").alias("n_hcps")
).orderBy("INDICATION_DECILE").show()

# ── Step 4: Drop Decile 1 ────────────────────────────────────────────────────────
hcp_nbrx_d2plus = hcp_nbrx.filter(F.col("INDICATION_DECILE") >= DECILE_CUTOFF)
print(f"HCPs before decile filter : {hcp_nbrx.count():,}")
print(f"HCPs after  decile filter : {hcp_nbrx_d2plus.count():,}")

In [0]:
# ── Step 5: Derive T1D / T2D / CKD / HF flags per HCP ────────────────────────
# An HCP gets a flag if ANY of their indication rows contains that indication label
hcp_ind_flags = lrx_ind.groupBy("PROVIDER_ID").agg(
    F.max(F.when(F.col(INDICATION_COL).contains("T1D"), 1).otherwise(0)).alias("T1D_FLAG"),
    F.max(F.when(F.col(INDICATION_COL).contains("T2D"), 1).otherwise(0)).alias("T2D_FLAG"),
    F.max(F.when(F.col(INDICATION_COL).contains("CKD"), 1).otherwise(0)).alias("CKD_FLAG"),
    F.max(F.when(F.col(INDICATION_COL).contains("HF"),  1).otherwise(0)).alias("HF_FLAG"),
)

# ── Step 6: Join decile + indication flags, add criterion flag ──────────────────
indication_hcps = (
    hcp_nbrx_d2plus
    .join(hcp_ind_flags, on="PROVIDER_ID", how="inner")
    .withColumn("INDICATION_HCP_FLAG", F.lit(1))
)

print(f"Indication-Based HCPs (Decile 2+): {indication_hcps.count():,}")
indication_hcps.agg(
    F.sum("T1D_FLAG").alias("T1D"),
    F.sum("T2D_FLAG").alias("T2D"),
    F.sum("CKD_FLAG").alias("CKD"),
    F.sum("HF_FLAG").alias("HF")
).show()

---
### Criterion 2 — Kerendia Writers

All HCPs who have ever prescribed Kerendia in **complete history** (no start-date filter, only `LOOKBACK_END` as upper bound). Sourced from `lrx_nbrx` using `PRODUCT_CLASS = 'KERENDIA'`.

In [0]:
# Full history Kerendia writers (no LOOKBACK_START filter)
kerendia_hcps = (
    lrx_nbrx
    .filter(
        (F.col("PRODUCT_CLASS") == "KERENDIA") &
        (F.col("WEEK_ENDING_DATE") <= LOOKBACK_END)
    )
    .select("PROVIDER_ID").distinct()
    .withColumn("KERENDIA_FLAG", F.lit(1))
)
print(f"Kerendia Writers: {kerendia_hcps.count():,}")

---
### Criterion 3 — Specialty HCPs

All HCPs with a target specialty within the lookback window, sourced **directly from `lrx_nbrx.SPECIALTY_GROUP`**.

This replaces the original 4-table MDM join chain:
~~`STG_CAR_IQV_LAAD_DIM_PRVDR → NPI XREF → MAP_CUSTOMER_XREF → MDM_STG_HCP_PRFL → specialty CSV`~~

Target specialties: **NEPH, CARD, ENDO, PCP, HOSPITALIST**

In [0]:
# ── Filter to target specialties in lookback window ─────────────────────────────
lrx_spec = lrx_window.filter(F.col("SPECIALTY_GROUP").isin(TARGET_SPECIALTIES))

# ── Pivot: one flag per specialty per PROVIDER_ID ─────────────────────────────
specialty_hcps = lrx_spec.groupBy("PROVIDER_ID").agg(
    F.max(F.when(F.col("SPECIALTY_GROUP") == "NEPH",        1).otherwise(0)).alias("NEPH_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "CARD",        1).otherwise(0)).alias("CARD_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "ENDO",        1).otherwise(0)).alias("ENDO_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "PCP",         1).otherwise(0)).alias("PCP_FLAG"),
    F.max(F.when(F.col("SPECIALTY_GROUP") == "HOSPITALIST", 1).otherwise(0)).alias("HOSPITALIST_FLAG"),
).withColumn("SPECIALTY_HCP_FLAG", F.lit(1))

print(f"Specialty HCPs total: {specialty_hcps.count():,}")
specialty_hcps.agg(
    F.sum("NEPH_FLAG").alias("NEPH"),
    F.sum("CARD_FLAG").alias("CARD"),
    F.sum("ENDO_FLAG").alias("ENDO"),
    F.sum("PCP_FLAG").alias("PCP"),
    F.sum("HOSPITALIST_FLAG").alias("HOSPITALIST")
).show()

---
### Criterion 4 — Targeting & BPT HCPs

All HCPs on the Targeting List or BPT List, sourced from `PHCDW.PHCDW_TARGETING.CP_KER_HCP`. Unchanged from v1.

In [0]:
# ── Targeting HCPs from CP_KER_HCP ───────────────────────────────────────────────
targeting_raw = Get_Data_Snowflakes("""
    SELECT DISTINCT BAYER_ID AS BH_ID
    FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
    WHERE FF IN ('CR1', 'CR2', 'CR4', 'CR5')
      AND (UPPER(PRODUCT_1) = 'KERENDIA' OR UPPER(PRODUCT_2) = 'KERENDIA')
""")

targeting_hcps = (
    targeting_raw
    .join(npi_provider_bhid_mapping.select("PROVIDER_ID", "BH_ID").distinct(), on="BH_ID", how="inner")
    .select("PROVIDER_ID").distinct()
    .withColumn("TARGETING_HCP_FLAG", F.lit(1))
)
print(f"Targeting HCPs: {targeting_hcps.count():,}")

In [0]:
# ── BPT HCPs from CP_KER_HCP (same source, separate flag) ─────────────────────
bpt_raw = Get_Data_Snowflakes("""
    SELECT DISTINCT BAYER_ID AS BH_ID
    FROM PHCDW.PHCDW_TARGETING.CP_KER_HCP
    WHERE FF IN ('CR1', 'CR2', 'CR4', 'CR5')
      AND (UPPER(PRODUCT_1) = 'KERENDIA' OR UPPER(PRODUCT_2) = 'KERENDIA')
""")

bpt_before = bpt_raw.count()
bpt_hcps = (
    bpt_raw
    .join(npi_provider_bhid_mapping.select("PROVIDER_ID", "BH_ID").distinct(), on="BH_ID", how="inner")
    .select("PROVIDER_ID").distinct()
    .withColumn("BPT_HCP_FLAG", F.lit(1))
)
bpt_after = bpt_hcps.count()
print(f"BPT HCPs before BH_ID mapping : {bpt_before:,}")
print(f"BPT HCPs after  BH_ID mapping : {bpt_after:,}")
print(f"Unmapped BH_IDs dropped       : {bpt_before - bpt_after:,}")

---
### Combining All Criteria

Full outer join on `PROVIDER_ID` across all 5 DataFrames. Fills nulls with 0 and deduplicates.

In [0]:
# ── Full outer join helper ──────────────────────────────────────────────────────────────
def outer_join(df_a: DataFrame, df_b: DataFrame) -> DataFrame:
    return df_a.join(df_b, on="PROVIDER_ID", how="outer")

# ── Columns to carry from indication_hcps into the join ─────────────────────────
# Keep decile and indication flags; drop TOTAL_NBRX (internal compute column)
indication_hcps_join = indication_hcps.drop("TOTAL_NBRX")

dfs = [
    indication_hcps_join,   # INDICATION_HCP_FLAG, T1D/T2D/CKD/HF_FLAG, INDICATION_DECILE
    kerendia_hcps,          # KERENDIA_FLAG
    specialty_hcps,         # SPECIALTY_HCP_FLAG, NEPH/CARD/ENDO/PCP/HOSPITALIST_FLAG
    targeting_hcps,         # TARGETING_HCP_FLAG
    bpt_hcps,               # BPT_HCP_FLAG
]

df_hcp_universe = reduce(outer_join, dfs)
df_hcp_universe = df_hcp_universe.fillna(0)
df_hcp_universe = df_hcp_universe.dropDuplicates(["PROVIDER_ID"])

print(f"Total HCP Universe: {df_hcp_universe.count():,}")
print(f"Columns: {df_hcp_universe.columns}")

In [0]:
# ── Aggregate flag totals ────────────────────────────────────────────────────────────
flag_cols = [c for c in df_hcp_universe.columns if c.endswith("_FLAG")]
agg_exprs = [F.sum(c).alias(c) for c in flag_cols]
flag_totals = df_hcp_universe.agg(*agg_exprs).collect()[0].asDict()

total = df_hcp_universe.count()
print(f"{'Criterion / Flag':<30} {'Count':>10} {'% of Universe':>15}")
print("-" * 58)
for col_name, cnt in sorted(flag_totals.items()):
    pct = 100 * int(cnt) / total
    label = col_name.replace("_FLAG", "").replace("_", " ")
    print(f"{label:<30} {int(cnt):>10,} {pct:>14.1f}%")
print("-" * 58)
print(f"{'TOTAL HCP UNIVERSE':<30} {total:>10,}")

In [0]:
# ── Preview final universe ─────────────────────────────────────────────────────────────
display(df_hcp_universe)

In [0]:
from datetime import datetime as dt

curr_date = dt.today().strftime("%Y_%m_%d")
print(f"Pull date: {curr_date}")

# ── Add pull date ───────────────────────────────────────────────────────────────
df_hcp_universe = df_hcp_universe.withColumn("PULL_DT", F.lit(curr_date))

# ── Write to Delta (overwrite) ─────────────────────────────────────────────────────
(
    df_hcp_universe
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(OUTPUT_PATH)
)
print(f"Saved to: {OUTPUT_PATH}")
print(f"Final HCP Universe size: {df_hcp_universe.count():,}")

---
### v1 vs v2 Universe Overlap Comparison

Loads both the original v1 (product-based) and new v2 (indication-based) HCP universes and computes overlap statistics.

In [0]:
# ── Load v1 and v2 universes ─────────────────────────────────────────────────────────
v1_path = (
    f"/Volumes/{CATALOG}/kerendia_nba_3/kerendia_nba_3_volume/"
    "hcp_level_model/dev_folder_3_1/notebook_output/"
    "kerendia_nba_3_hcp_model_base_hcp_universe_for_inference"
)
v2_path = OUTPUT_PATH

v1_universe = spark.read.format("delta").load(v1_path).select("PROVIDER_ID").distinct()
v2_universe = spark.read.format("delta").load(v2_path).select("PROVIDER_ID").distinct()

v1_count = v1_universe.count()
v2_count = v2_universe.count()

# ── Overlap ───────────────────────────────────────────────────────────────────────────────
overlap = v1_universe.join(v2_universe, on="PROVIDER_ID", how="inner")
overlap_count = overlap.count()

v1_only = v1_universe.join(v2_universe, on="PROVIDER_ID", how="left_anti")
v1_only_count = v1_only.count()

v2_only = v2_universe.join(v1_universe, on="PROVIDER_ID", how="left_anti")
v2_only_count = v2_only.count()

print(f"v1 Universe (product-based) : {v1_count:,}")
print(f"v2 Universe (indication-based): {v2_count:,}")
print(f"Overlap (in both)            : {overlap_count:,}")
print(f"v1 only (dropped in v2)      : {v1_only_count:,}  ({100*v1_only_count/v1_count:.1f}% of v1)")
print(f"v2 only (new in v2)          : {v2_only_count:,}  ({100*v2_only_count/v2_count:.1f}% of v2)")
print(f"v1 retention in v2           : {100*overlap_count/v1_count:.1f}%")
print(f"v2 growth over v1            : {100*(v2_count - v1_count)/v1_count:+.1f}%")

In [0]:
# ── Summary table ─────────────────────────────────────────────────────────────────────
from pyspark.sql import Row

summary = spark.createDataFrame([
    Row(Metric="v1 Universe (product-based)", Count=v1_count),
    Row(Metric="v2 Universe (indication-based)", Count=v2_count),
    Row(Metric="Overlap (in both)", Count=overlap_count),
    Row(Metric="v1 only (dropped in v2)", Count=v1_only_count),
    Row(Metric="v2 only (new in v2)", Count=v2_only_count),
    Row(Metric="v1 retention in v2 (%)", Count=int(100*overlap_count/v1_count)),
    Row(Metric="v2 growth over v1 (%)", Count=int(100*(v2_count - v1_count)/v1_count)),
])
display(summary)

---
### Investigating v1-only HCPs (dropped in v2)

These 1,454 HCPs qualified for v1 but not v2. We need to understand:
1. Which v1 criteria did they qualify under?
2. Do they appear in `lrx_nbrx` at all? If so, what indication / specialty / decile?
3. Are they Targeting/BPT-only HCPs with no `lrx_nbrx` presence?

In [0]:
# ── Load full v1 universe with all flags ───────────────────────────────────────
v1_full = spark.read.format("delta").load(v1_path)

# ── Filter to v1-only HCPs ─────────────────────────────────────────────────────
v1_only_full = v1_full.join(v2_universe, on="PROVIDER_ID", how="left_anti")

print(f"v1-only HCPs: {v1_only_full.count():,}")
print(f"v1 columns: {v1_full.columns}")
v1_only_full.printSchema()

In [0]:
# ── Which v1 criteria do the dropped HCPs qualify under? ───────────────────────
# v1 flag columns (may differ from v2)
v1_flag_cols = [c for c in v1_only_full.columns if c.endswith("_FLAG") or c in ("SGLT2_GLPI_FLAG", "KERENDIA_FLAG", "NEPH_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG")]
print(f"v1 flag columns: {v1_flag_cols}\n")

agg_exprs = [F.sum(c).alias(c) for c in v1_flag_cols]
print("v1-only HCPs by criterion flag:")
v1_only_full.agg(*agg_exprs).display()

In [0]:
# ── Check if v1-only HCPs appear in lrx_nbrx at all ─────────────────────────────
v1_only_ids = v1_only_full.select("PROVIDER_ID").distinct()

# In the lookback window
in_lrx_window = lrx_window.join(v1_only_ids, on="PROVIDER_ID", how="inner").select("PROVIDER_ID").distinct()
in_window_count = in_lrx_window.count()

# In full lrx_nbrx history
in_lrx_full = lrx_nbrx.join(v1_only_ids, on="PROVIDER_ID", how="inner").select("PROVIDER_ID").distinct()
in_full_count = in_lrx_full.count()

# Not in lrx_nbrx at all
not_in_lrx = v1_only_ids.join(lrx_nbrx, on="PROVIDER_ID", how="left_anti")
not_in_lrx_count = not_in_lrx.count()

print(f"v1-only HCPs in lrx_window     : {in_window_count:,}")
print(f"v1-only HCPs in full lrx_nbrx   : {in_full_count:,}")
print(f"v1-only HCPs NOT in lrx_nbrx   : {not_in_lrx_count:,}")

In [0]:
# ── For v1-only HCPs that ARE in lrx_window, check their indication & specialty ─────
v1_only_in_window = lrx_window.join(v1_only_ids, on="PROVIDER_ID", how="inner")

print("INDICATION_GROUP_APP1 distribution for v1-only HCPs in window:")
v1_only_in_window.groupBy(INDICATION_COL).agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps")
).orderBy(F.desc("n_hcps")).show(20, truncate=False)

print("SPECIALTY_GROUP distribution for v1-only HCPs in window:")
v1_only_in_window.groupBy("SPECIALTY_GROUP").agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps")
).orderBy(F.desc("n_hcps")).show(20, truncate=False)

In [0]:
# ── For v1-only HCPs in lrx_window with non-OTHERS indications, check decile ─────
# These are HCPs who SHOULD have been captured by Criterion 1 but weren't
v1_only_ind = v1_only_in_window.filter(F.col(INDICATION_COL) != "OTHERS")
v1_only_ind_hcps = v1_only_ind.groupBy("PROVIDER_ID").agg(
    F.sum("NBRX").alias("TOTAL_NBRX")
)

# Assign decile using the SAME window as Criterion 1 (all indication HCPs)
# to see where these dropped HCPs fall
v1_only_with_decile = v1_only_ind_hcps.join(
    hcp_nbrx.select("PROVIDER_ID", "INDICATION_DECILE"),
    on="PROVIDER_ID", how="left"
)

print("Decile distribution of v1-only HCPs (among indication pool):")
v1_only_with_decile.groupBy("INDICATION_DECILE").agg(
    F.count("*").alias("n_hcps")
).orderBy("INDICATION_DECILE").show()

# How many are in Decile 1 (the dropped decile)?
decile1_count = v1_only_with_decile.filter(F.col("INDICATION_DECILE") == 1).count()
print(f"v1-only HCPs in Decile 1 (dropped by Criterion 1 filter): {decile1_count:,}")
print(f"v1-only HCPs with non-OTHERS indications total           : {v1_only_ind_hcps.count():,}")

In [0]:
# ── Summary: why each v1-only HCP was dropped ─────────────────────────────────
# Re-join v1-only IDs with lrx_window to get indication & specialty columns
v1_only_in_window_full = lrx_window.join(v1_only_ids, on="PROVIDER_ID", how="inner")

# Category 1: Not in lrx_nbrx at all
cat_not_in_lrx = not_in_lrx.withColumn("DROP_REASON", F.lit("Not in lrx_nbrx at all"))

# Category 2: In lrx_window but only OTHERS indications & not in target specialties
cat_others_only = (
    v1_only_in_window_full
    .groupBy("PROVIDER_ID")
    .agg(
        F.max(F.when(F.col(INDICATION_COL) != "OTHERS", 1).otherwise(0)).alias("has_non_others"),
        F.max(F.when(F.col("SPECIALTY_GROUP").isin(TARGET_SPECIALTIES), 1).otherwise(0)).alias("has_target_spec"),
    )
    .filter((F.col("has_non_others") == 0) & (F.col("has_target_spec") == 0))
    .withColumn("DROP_REASON", F.lit("Only OTHERS indications & no target specialty"))
)

# Category 3: Has non-OTHERS indications but fell in Decile 1
cat_decile1 = (
    v1_only_with_decile
    .filter(F.col("INDICATION_DECILE") == 1)
    .withColumn("DROP_REASON", F.lit("Indication HCP but Decile 1 (low volume)"))
)

# Category 4: Has non-OTHERS indications, Decile 2+, but not in target specialty (edge case)
cat_other = (
    v1_only_with_decile
    .filter(F.col("INDICATION_DECILE") >= DECILE_CUTOFF)
    .withColumn("DROP_REASON", F.lit("Decile 2+ indication HCP — should be in v2 (edge case)"))
)

# Combine all categories
drop_reasons = (
    cat_not_in_lrx.select("PROVIDER_ID", "DROP_REASON")
    .unionByName(cat_others_only.select("PROVIDER_ID", "DROP_REASON"))
    .unionByName(cat_decile1.select("PROVIDER_ID", "DROP_REASON"))
    .unionByName(cat_other.select("PROVIDER_ID", "DROP_REASON"))
)

# Category 5: In lrx_nbrx but outside lookback window (the largest group)
cat_outside_window = (
    in_lrx_full
    .join(in_lrx_window, on="PROVIDER_ID", how="left_anti")
    .withColumn("DROP_REASON", F.lit("In lrx_nbrx but outside lookback window"))
)
drop_reasons = drop_reasons.unionByName(cat_outside_window.select("PROVIDER_ID", "DROP_REASON"))

print("Drop reason summary:")
drop_reasons.groupBy("DROP_REASON").agg(
    F.count("*").alias("n_hcps")
).orderBy(F.desc("n_hcps")).show(truncate=False)

print(f"Total categorized: {drop_reasons.count():,} (should be ~{v1_only_count:,})")

---
### v2-only HCPs by Specialty

These 854,967 HCPs are **new** in v2 (not in v1). Profile them by specialty, indication flags, and which v2 criterion brought them in.

In [0]:
# ── Load v2-only HCPs with full v2 flags ─────────────────────────────────────────
v2_full = spark.read.format("delta").load(v2_path)
v2_only_full = v2_full.join(v1_universe, on="PROVIDER_ID", how="left_anti")

v2_only_count = v2_only_full.count()
print(f"v2-only HCPs: {v2_only_count:,}\n")

# ── Specialty distribution ─────────────────────────────────────────────────────────
spec_cols = ["NEPH_FLAG", "CARD_FLAG", "ENDO_FLAG", "PCP_FLAG", "HOSPITALIST_FLAG"]
print(f"{'Specialty':<20} {'v2-only':>10} {'% of v2-only':>14} {'v2 total':>10} {'% new':>8}")
print("-" * 65)

# Compute v2-total per specialty for comparison
v2_total_specs = v2_full.agg(*[F.sum(c).alias(c) for c in spec_cols]).collect()[0].asDict()
v2_only_specs  = v2_only_full.agg(*[F.sum(c).alias(c) for c in spec_cols]).collect()[0].asDict()

for c in spec_cols:
    label = c.replace("_FLAG", "")
    new_n = int(v2_only_specs[c])
    tot_n = int(v2_total_specs[c])
    pct_of_v2only = 100 * new_n / v2_only_count if v2_only_count else 0
    pct_new       = 100 * new_n / tot_n if tot_n else 0
    print(f"{label:<20} {new_n:>10,} {pct_of_v2only:>13.1f}% {tot_n:>10,} {pct_new:>7.1f}%")

# No target specialty at all
no_spec = v2_only_full.filter(F.col("SPECIALTY_HCP_FLAG") == 0).count()
print(f"{'(no target spec)':<20} {no_spec:>10,} {100*no_spec/v2_only_count:>13.1f}%")

In [0]:
# ── Which v2 criterion brought each v2-only HCP in? ─────────────────────────────
# An HCP may qualify under multiple criteria
criterion_cols = ["INDICATION_HCP_FLAG", "KERENDIA_FLAG", "SPECIALTY_HCP_FLAG", "TARGETING_HCP_FLAG", "BPT_HCP_FLAG"]

print(f"{'Criterion':<25} {'v2-only':>10} {'% of v2-only':>14}")
print("-" * 52)
for c in criterion_cols:
    n = v2_only_full.filter(F.col(c) == 1).count()
    pct = 100 * n / v2_only_count
    label = c.replace("_FLAG", "").replace("_", " ")
    print(f"{label:<25} {n:>10,} {pct:>13.1f}%")

# Criterion combinations for v2-only
print("\nTop criterion combinations (v2-only):")
v2_only_full.groupBy(*criterion_cols).agg(
    F.count("*").alias("n_hcps")
).orderBy(F.desc("n_hcps")).show(15)

In [0]:
# ── Indication flag distribution for v2-only HCPs ────────────────────────────────
ind_cols = ["T1D_FLAG", "T2D_FLAG", "CKD_FLAG", "HF_FLAG"]

print(f"{'Indication':<15} {'v2-only':>10} {'% of v2-only':>14} {'v2 total':>10} {'% new':>8}")
print("-" * 60)

v2_total_inds = v2_full.agg(*[F.sum(c).alias(c) for c in ind_cols]).collect()[0].asDict()
v2_only_inds  = v2_only_full.agg(*[F.sum(c).alias(c) for c in ind_cols]).collect()[0].asDict()

for c in ind_cols:
    label = c.replace("_FLAG", "")
    new_n = int(v2_only_inds[c])
    tot_n = int(v2_total_inds[c])
    pct_of_v2only = 100 * new_n / v2_only_count if v2_only_count else 0
    pct_new       = 100 * new_n / tot_n if tot_n else 0
    print(f"{label:<15} {new_n:>10,} {pct_of_v2only:>13.1f}% {tot_n:>10,} {pct_new:>7.1f}%")

# No indication at all (specialty-only or targeting-only)
no_ind = v2_only_full.filter(F.col("INDICATION_HCP_FLAG") == 0).count()
print(f"{'(no indication)':<15} {no_ind:>10,} {pct_of_v2only:>13.1f}%")

In [0]:
# ── Decile distribution of v2-only indication HCPs ────────────────────────────────
print("Decile distribution of v2-only HCPs (indication-based only):")
v2_only_full.filter(F.col("INDICATION_HCP_FLAG") == 1).groupBy("INDICATION_DECILE").agg(
    F.count("*").alias("n_hcps")
).orderBy("INDICATION_DECILE").show()

---
### Profiling the 274K No-Specialty Indication HCPs

These HCPs entered v2 via Criterion 1 (Indication Decile 2+) but have **no target specialty** (NEPH/CARD/ENDO/PCP/HOSPITALIST). Their `SPECIALTY_GROUP` in `lrx_nbrx` is either `OTHERS` or `HOS`. This section profiles their indication mix, decile distribution, drug class activity, and actual specialty values.

In [0]:
# ── v2-only HCPs with INDICATION_HCP_FLAG=1 but SPECIALTY_HCP_FLAG=0 ───────────
no_spec_ind = v2_only_full.filter(
    (F.col("INDICATION_HCP_FLAG") == 1) & (F.col("SPECIALTY_HCP_FLAG") == 0)
)
no_spec_count = no_spec_ind.count()
print(f"No-specialty indication HCPs (v2-only): {no_spec_count:,}")

In [0]:
# ── Indication flags ─────────────────────────────────────────────────────────────────
ind_cols = ["T1D_FLAG", "T2D_FLAG", "CKD_FLAG", "HF_FLAG"]
ind_agg = no_spec_ind.agg(*[F.sum(c).alias(c) for c in ind_cols]).collect()[0].asDict()

print(f"{'Indication':<12} {'Count':>10} {'%':>8}")
print("-" * 32)
for c in ind_cols:
    n = int(ind_agg[c])
    print(f"{c.replace('_FLAG',''):<12} {n:>10,} {100*n/no_spec_count:>7.1f}%")

# ── Top indication combinations ────────────────────────────────────────────────────
print("\nTop indication combinations:")
no_spec_ind.groupBy(*ind_cols).agg(
    F.count("*").alias("n_hcps")
).orderBy(F.desc("n_hcps")).show(15)

# ── Decile distribution ───────────────────────────────────────────────────────────
print("Decile distribution:")
no_spec_ind.groupBy("INDICATION_DECILE").agg(
    F.count("*").alias("n_hcps")
).orderBy("INDICATION_DECILE").show()

In [0]:
# ── What SPECIALTY_GROUP do these HCPs actually have in lrx_nbrx? ───────────────
no_spec_ids = no_spec_ind.select("PROVIDER_ID")
no_spec_lrx = lrx_window.join(no_spec_ids, on="PROVIDER_ID", how="inner")

print("SPECIALTY_GROUP distribution for no-spec indication HCPs:")
no_spec_lrx.groupBy("SPECIALTY_GROUP").agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps")
).orderBy(F.desc("n_hcps")).show(truncate=False)

In [0]:
# ── What INDICATION_GROUP_APP1 values do they prescribe for? ───────────────────
print("INDICATION_GROUP_APP1 distribution:")
no_spec_lrx.groupBy(INDICATION_COL).agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps"),
    F.sum("NBRX").alias("total_nbrx")
).orderBy(F.desc("n_hcps")).show(20, truncate=False)

In [0]:
# ── What drug classes are these HCPs prescribing? ───────────────────────────────
print("Top PRODUCT_CLASS for no-spec indication HCPs:")
no_spec_lrx.groupBy("PRODUCT_CLASS").agg(
    F.countDistinct("PROVIDER_ID").alias("n_hcps"),
    F.sum("NBRX").alias("total_nbrx")
).orderBy(F.desc("n_hcps")).show(20, truncate=False)

In [0]:
# ── How many of these no-spec HCPs are Kerendia writers? ────────────────────────
ker_in_nospec = no_spec_ind.filter(F.col("KERENDIA_FLAG") == 1).count()
print(f"Kerendia writers among no-spec indication HCPs: {ker_in_nospec:,} ({100*ker_in_nospec/no_spec_count:.2f}%)")

# ── Summary stats ──────────────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"SUMMARY: 274K No-Specialty Indication HCPs")
print(f"{'='*60}")
print(f"Total count              : {no_spec_count:,}")
print(f"% of v2-only (855K)      : {100*no_spec_count/v2_only_count:.1f}%")
print(f"Kerendia writers         : {ker_in_nospec:,}")
print(f"All are SPECIALTY_GROUP  = 'OTHERS' or 'HOS' in lrx_nbrx")
print(f"Entered v2 via Criterion 1 (Indication Decile 2+) only")

---
### v2-only HCPs by Product Group

Profiles the 854,967 net-new HCPs by the drug product groups they prescribe, showing HCP counts, NBRx volume, and Kerendia overlap.

In [0]:
# ── v2-only HCPs' PRODUCT_GROUP distribution ───────────────────────────────────
v2_only_ids = v2_only_full.select("PROVIDER_ID")
v2_only_lrx = lrx_window.join(v2_only_ids, on="PROVIDER_ID", how="inner")

print(f"{'PRODUCT_GROUP':<30} {'HCPs':>10} {'Total NBRx':>12} {'Kerendia HCPs':>15}")
print("-" * 70)

# Aggregate by PRODUCT_GROUP
pg_stats = (
    v2_only_lrx
    .groupBy("PRODUCT_GROUP")
    .agg(
        F.countDistinct("PROVIDER_ID").alias("n_hcps"),
        F.sum("NBRX").alias("total_nbrx"),
    )
)

# Also compute Kerendia overlap per product group: HCPs who prescribe both that product AND Kerendia
ker_ids = v2_only_lrx.filter(F.col("PRODUCT_CLASS") == "KERENDIA").select("PROVIDER_ID").distinct()
pg_ker = v2_only_lrx.join(ker_ids, on="PROVIDER_ID", how="inner").groupBy("PRODUCT_GROUP").agg(
    F.countDistinct("PROVIDER_ID").alias("ker_hcps")
)

pg_final = pg_stats.join(pg_ker, on="PRODUCT_GROUP", how="left").orderBy(F.desc("n_hcps"))

for row in pg_final.collect():
    pg = row["PRODUCT_GROUP"] or "(null)"
    n = row["n_hcps"]
    nbrx = row["total_nbrx"] or 0
    ker = row["ker_hcps"] or 0
    print(f"{pg:<30} {n:>10,} {int(nbrx):>12,} {ker:>15,}")

print(f"\nTotal v2-only HCPs with lrx_window activity: {v2_only_lrx.select('PROVIDER_ID').distinct().count():,}")

In [0]:
# ── Product Group breakdown by v2 criterion cohort ────────────────────────────────
# Split v2-only into: (a) Indication+Specialty, (b) Indication-only, (c) Specialty-only
ind_and_spec = v2_only_full.filter((F.col("INDICATION_HCP_FLAG") == 1) & (F.col("SPECIALTY_HCP_FLAG") == 1)).select("PROVIDER_ID")
ind_only     = v2_only_full.filter((F.col("INDICATION_HCP_FLAG") == 1) & (F.col("SPECIALTY_HCP_FLAG") == 0)).select("PROVIDER_ID")
spec_only    = v2_only_full.filter((F.col("INDICATION_HCP_FLAG") == 0) & (F.col("SPECIALTY_HCP_FLAG") == 1)).select("PROVIDER_ID")

cohorts = [("Indication+Specialty", ind_and_spec), ("Indication-only", ind_only), ("Specialty-only", spec_only)]

for label, ids in cohorts:
    n_cohort = ids.count()
    lrx_cohort = lrx_window.join(ids, on="PROVIDER_ID", how="inner")
    print(f"\n{'='*70}");
    print(f"{label} ({n_cohort:,} HCPs) — Top 10 PRODUCT_GROUP:")
    print(f"{'PRODUCT_GROUP':<30} {'HCPs':>10} {'NBRx':>12} {'% HCPs':>8}")
    print("-" * 63)
    pg_c = lrx_cohort.groupBy("PRODUCT_GROUP").agg(
        F.countDistinct("PROVIDER_ID").alias("n_hcps"),
        F.sum("NBRX").alias("total_nbrx"),
    ).orderBy(F.desc("n_hcps")).limit(10)
    for row in pg_c.collect():
        pg = row["PRODUCT_GROUP"] or "(null)"
        n = row["n_hcps"]
        nbrx = row["total_nbrx"] or 0
        pct = 100 * n / n_cohort if n_cohort else 0
        print(f"{pg:<30} {n:>10,} {int(nbrx):>12,} {pct:>7.1f}%")